# Fase 1 — Análise Exploratória (EDA)

**Tema 08:** RH e People Analytics  
**Objetivo desta aula/fase:** entender a base, gerar insights de negócio e sustentar hipóteses para a clusterização (Fase 3).

**Entrada:** `data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv`  
**Complementa:** `01_exploracao_inicial.ipynb`  
**Dicionário:** `references/dicionario_dados.md`

Estrutura em **8 blocos** (para o Cap. III do relatório).


## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_RAW / "WA_Fn-UseC_-HR-Employee-Attrition.csv")
print(df.shape)
df.head(3)

---
## Bloco 1 — Estrutura da base
Dimensão, tipos e papel de cada variável no problema de RH.

In [ ]:
likert_cols = [
    "EnvironmentSatisfaction",
    "JobSatisfaction",
    "RelationshipSatisfaction",
    "JobInvolvement",
    "WorkLifeBalance",
]

cat_cols = df.select_dtypes(include="object").columns.tolist()
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
print(f"\nCategóricas ({len(cat_cols)}): {cat_cols}")
print(f"\nNuméricas ({len(num_cols)}): {num_cols}")
print(f"\nLikert (satisfação/engajamento): {likert_cols}")

pd.DataFrame(
    {
        "coluna": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "n_unicos": [df[c].nunique() for c in df.columns],
        "n_ausentes": df.isnull().sum().values,
    }
)

---
## Bloco 2 — Qualidade dos dados
Ausentes, duplicatas, colunas constantes e IDs.

In [ ]:
constantes = [c for c in df.columns if df[c].nunique(dropna=False) == 1]

print("Ausentes totais:", int(df.isnull().sum().sum()))
print("Linhas duplicadas:", int(df.duplicated().sum()))
print("EmployeeNumber duplicados:", int(df["EmployeeNumber"].duplicated().sum()))
print("Colunas constantes:", constantes)

for c in constantes:
    print(f"  - {c} = {df[c].iloc[0]!r}")

---
## Bloco 3 — Análise univariada (numéricas contínuas / de carreira)
Foco em idade, renda, tempo de casa e experiência.

In [ ]:
vars_carreira = [
    "Age",
    "MonthlyIncome",
    "TotalWorkingYears",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "DistanceFromHome",
    "PercentSalaryHike",
]

df[vars_carreira].describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()

for ax, col in zip(axes, vars_carreira):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(col)

plt.tight_layout()
plt.savefig(FIG_DIR / "01_hist_carreira.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Bloco 4 — Análise univariada (categóricas e Likert)
Departamento, cargo, horas extras e escalas de satisfação.

In [ ]:
cats_negocio = [
    "Department",
    "JobRole",
    "BusinessTravel",
    "OverTime",
    "MaritalStatus",
    "Gender",
    "EducationField",
]

for col in cats_negocio:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(normalize=True).mul(100).round(1).astype(str) + "%")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

for ax, col in zip(axes, likert_cols):
    ordem = sorted(df[col].unique())
    sns.countplot(data=df, x=col, order=ordem, ax=ax, color="teal")
    ax.set_title(col)

plt.tight_layout()
plt.savefig(FIG_DIR / "02_likert.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Bloco 5 — Variável de retenção (Attrition) e desempenho
`Attrition` e `PerformanceRating` serão **reservadas para avaliação** na modelagem; aqui usamos para entender o negócio.

In [ ]:
attr = (
    df["Attrition"]
    .value_counts()
    .to_frame("n")
    .assign(pct=lambda x: (x["n"] / x["n"].sum() * 100).round(2))
)
print("Attrition")
display(attr)

print("\nPerformanceRating")
display(df["PerformanceRating"].value_counts().sort_index())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(
    data=df,
    x="Attrition",
    order=["No", "Yes"],
    hue="Attrition",
    ax=ax,
    palette=["#4C78A8", "#E45756"],
    legend=False,
)
ax.set_title("Distribuição de Attrition")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_attrition.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Bloco 6 — Análise bivariada com Attrition
Quais fatores aparecem associados à saída?

In [ ]:
def taxa_attrition(coluna, dados=df):
    tab = (
        dados.groupby(coluna)["Attrition"]
        .apply(lambda s: (s == "Yes").mean() * 100)
        .sort_values(ascending=False)
        .round(2)
        .rename("% Attrition")
    )
    return tab


for col in ["OverTime", "MaritalStatus", "BusinessTravel", "Department", "JobRole", "Gender"]:
    print(f"\n=== Taxa de saída por {col} ===")
    print(taxa_attrition(col))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ["MonthlyIncome", "Age", "YearsAtCompany"]):
    sns.boxplot(data=df, x="Attrition", y=col, order=["No", "Yes"], ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.savefig(FIG_DIR / "04_boxplot_attrition.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ["JobSatisfaction", "EnvironmentSatisfaction", "WorkLifeBalance"]):
    tab = taxa_attrition(col).sort_index()
    tab.plot(kind="bar", ax=ax, color="#E45756", rot=0)
    ax.set_ylabel("% Attrition")
    ax.set_title(col)

plt.tight_layout()
plt.savefig(FIG_DIR / "05_likert_vs_attrition.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Bloco 7 — Correlações e possíveis multicolinearidades
Útil para PCA e para evitar variáveis quase redundantes na clusterização.

In [ ]:
corr_vars = [
    "Age",
    "MonthlyIncome",
    "JobLevel",
    "TotalWorkingYears",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsWithCurrManager",
    "YearsSinceLastPromotion",
    "PercentSalaryHike",
    "PerformanceRating",
    "JobSatisfaction",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "WorkLifeBalance",
    "DistanceFromHome",
    "NumCompaniesWorked",
]

corr = df[corr_vars].corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=False, square=True)
plt.title("Correlação entre variáveis numéricas selecionadas")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_correlacao.png", dpi=120, bbox_inches="tight")
plt.show()

# Pares mais correlacionados (|r| >= 0.60), sem diagonal
pares = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().rename("r").reset_index()
pares.columns = ["var1", "var2", "r"]
pares["abs_r"] = pares["r"].abs()
pares.sort_values("abs_r", ascending=False).query("abs_r >= 0.60").head(15)

---
## Bloco 8 — Insights e hipóteses para a próxima fase
Responda às perguntas do Canvas com evidência exploratória (ainda sem clusters).

In [ ]:
# Resumo rápido para embasar os insights
resumo = {
    "taxa_attrition_geral_%": round((df["Attrition"] == "Yes").mean() * 100, 2),
    "attrition_com_overtime_%": round(
        (df.loc[df["OverTime"] == "Yes", "Attrition"] == "Yes").mean() * 100, 2
    ),
    "attrition_sem_overtime_%": round(
        (df.loc[df["OverTime"] == "No", "Attrition"] == "Yes").mean() * 100, 2
    ),
    "renda_media_sair": round(df.loc[df["Attrition"] == "Yes", "MonthlyIncome"].mean(), 2),
    "renda_media_ficar": round(df.loc[df["Attrition"] == "No", "MonthlyIncome"].mean(), 2),
    "idade_media_sair": round(df.loc[df["Attrition"] == "Yes", "Age"].mean(), 2),
    "idade_media_ficar": round(df.loc[df["Attrition"] == "No", "Age"].mean(), 2),
}
pd.Series(resumo).to_frame("valor")

### Coleção de insights (edite após rodar as células)

Use os números gerados acima. Sugestão inicial de preenchimento:

1. **Retenção desigual:** a taxa de attrition não é uniforme — OverTime e estado civil tendem a diferenciar risco.
2. **Horas extras:** colaboradores com `OverTime = Yes` apresentam taxa de saída bem maior → hipótese do Grupo "Alta carga".
3. **Renda e junioridade:** quem sai tende a ter renda e idade médias menores → hipótese de perfil início de carreira / sub-remunerado.
4. **Satisfação:** níveis baixos nas escalas Likert associam-se a mais saída → útil para trilhas de engajamento.
5. **Multicolinearidade:** `JobLevel`, `MonthlyIncome` e `TotalWorkingYears` caminham juntos → na preparação, considerar se PCA ou seleção de features evita redundância.
6. **Colunas inúteis para modelo:** `EmployeeCount`, `StandardHours`, `Over18` são constantes; `EmployeeNumber` é ID.
7. **Base sintética:** estrutura de clusters pode ser fraca (aviso do PDF) — sucesso = interpretabilidade + ação, não só silhueta alta.
8. **Próximo passo (Fase 2):** remover constantes/ID, separar features de clustering vs avaliação, salvar base preparada.

### Hipóteses a testar na clusterização

| Hipótese (Canvas) | Sinal na EDA | Status |
|---|---|---|
| Existe grupo de alta carga / overtime | OverTime eleva attrition | a validar com cluster |
| Existe grupo estável/engajado | Baixa saída + alta satisfação | a validar |
| Existe grupo de risco (baixa satisfação/renda) | Renda menor entre quem sai | a validar |
| Existe perfil início de carreira | Idade/anos menores entre quem sai | a validar |
| No máximo 5 grupos acionáveis | restrição da diretoria | regra de modelagem |

## Checklist Fase 1

- [x] Estrutura e tipos
- [x] Qualidade (ausentes, constantes, ID)
- [x] Univariada numérica
- [x] Univariada categórica/Likert
- [x] Attrition e Performance (visão de negócio)
- [x] Bivariada com attrition
- [x] Correlações
- [x] Insights / hipóteses

**Figuras salvas em:** `reports/figures/`  
**Seguir para:** `03_preparacao.ipynb` (Fase 2)